# Contact Centre Fraud — Synthetic Data Generator & Detection Rules

Generates seven relational tables of contact-centre activity for an Australian government service agency, then recovers the fraud with eight detection rules — seven of them a single query, the eighth a volume-adjusted overlap score.

**The design constraint: there are no fraud flags anywhere in the output.** No `fraud_label`, no `scenario_type`, no `risk_score`. Every row is an ordinary business record. Fraud exists only as a *pattern across rows* — and Part 2 of this notebook proves each pattern is findable.

Three principles make that work:

1. **One code path.** A fraudulent CRM lookup is built by the same function as a legitimate one. The generator never branches on "is this fraud" when writing a row — only on which arguments to pass.
2. **Hidden propensity.** A small share of agents carry a latent `_dishonest` marker that steers which ordinary records get generated. It is dropped before export and never reaches disk.
3. **Base rates.** Every artefact that appears in a fraud pattern also appears in volume in legitimate activity — refunds, failed logins, after-hours access and no-contact lookups are all common and mostly innocent. No single column gives the answer away.

**Structure:** Part 1 (Sections 0–5) generates and exports the data. Part 2 (Sections 6–7) reloads the CSVs and runs the rules to confirm fraud cases.

Companion files: `README.md` (design rationale and demo narrative), `schema.mermaid` (ERD), `data_dictionary.md` (field definitions).

## Part 1 — Generation

### Section 0 · Configuration

Every knob lives here. The parameters split into three groups, and the split *is* the design:

- **Volume** — rescale the dataset without touching logic.
- **Legitimate base rates** (`P_LEGIT_*`, `P_SELF_SERVE_*`) — these are what stop the detection rules from being trivial. Raise them and the noise floor climbs toward the fraud signal; the dataset gets harder.
- **Fraud pattern intensity** (`*_PER_MONTH`, burst sizes) — how busy each offender is. Raise these and the dataset gets easier.

`FRAUD_TREND_YOY` grows offending 15% year on year, so a by-year chart has a story. `SEED` makes every run reproducible.

In [1]:
import os
import random
from collections import defaultdict
from datetime import datetime, timedelta

import pandas as pd

In [2]:
SEED = 42
OUTPUT_DIR = "output"

# --- Volume -----------------------------------------------------------------
NUM_CUSTOMERS = 7500
NUM_AGENTS = 20
CONTACTS_PER_DAY = 150
YEARS = 3
START_DATE = datetime(2023, 1, 1)

# --- Channel mix ------------------------------------------------------------
P_SELF_SERVICE = 0.45

# --- Legitimate base rates (these are what stop the rules from being trivial) -
P_CONTACT_HAS_TRANSACTION = 0.22   # contacts that produce a transaction
P_CONTACT_HAS_PROFILE_CHANGE = 0.09  # contacts that update the customer record
P_TXN_IS_REFUND = 0.07             # refunds/waivers as a share of transactions
P_LEGIT_REFUND_HAS_PURCHASE = 0.94  # honest refunds nearly always follow a purchase
P_LEGIT_NO_CONTACT_ACCESS = 0.008   # QA review, callback prep, complaint handling
P_LEGIT_WARM_TRANSFER = 0.006       # agent opens a different customer, legitimately
P_LEGIT_3_FIELD_CHANGE = 0.015      # honest bulk update (new identity documents)
P_SELF_SERVE_FAILED_LOGIN = 0.14    # sessions with 1-2 fumbled password attempts
P_SELF_SERVE_LOCKOUT = 0.020        # honest sessions with 3+ failures then success

# --- Hidden agent propensity (never exported) --------------------------------
P_AGENT_DISHONEST = 0.15    # share of agents who will offend
P_AGENT_TERMINATED = 0.11   # share of agents who leave during the window

# --- Fraud pattern intensity -------------------------------------------------
LOCATE_LOOKUPS_PER_MONTH = 12      # unlawful locate events per dishonest agent
P_LOCATE_AFTER_HOURS = 0.35        # of those, share done outside rostered shift
VELOCITY_BURSTS_PER_YEAR = 3       # excessive-lookup episodes per dishonest agent
VELOCITY_BURST_SIZE = (22, 40)     # customers pulled in a single hour
BENEFIT_FRAUD_PER_MONTH = 2        # bogus refunds/waivers per dishonest agent
IDENTITY_FRAUD_PER_MONTH = 1       # takeover-at-the-counter events per agent
ACCOUNT_TAKEOVERS_PER_MONTH = 6    # external self-service takeovers, agency-wide
POST_TERMINATION_ACCESSES = (3, 9) # lookups by a leaver after their last day

# --- Year-on-year growth in offending ----------------------------------------
FRAUD_TREND_YOY = 0.15

# --- Business calendar --------------------------------------------------------
SHIFT_WINDOWS = {"day": (7, 15), "evening": (15, 23), "night": (23, 7)}

TOTAL_DAYS = 365 * YEARS
DATE_RANGE = [START_DATE + timedelta(days=i) for i in range(TOTAL_DAYS)]

random.seed(SEED)

### Section 1 · Reference pools & helpers

Names and addresses are assembled from independent component pools (first × last, street × suburb), so the combination space is the product of the pools rather than the size of any one — this is what keeps 7,500 customers from looking repetitive.

Two helpers matter later: `in_shift()` / `shift_hour()` encode the roster windows that Rule 2 (after-hours access) checks against, and `year_weight()` applies the year-on-year offending trend *everywhere* it's used — every fraud pattern draws its daily probability through it, so the trend holds across all scenarios.

In [3]:
FIRST_NAMES = [
    "Oliver", "Charlotte", "William", "Amelia", "Jack", "Isla", "Noah", "Mia",
    "Thomas", "Grace", "James", "Ava", "Lucas", "Chloe", "Henry", "Sophie",
    "Ethan", "Zoe", "Liam", "Ruby", "Alexander", "Evie", "Samuel", "Harper",
    "Benjamin", "Willow", "Hugo", "Ella", "Archie", "Matilda", "Leo", "Poppy",
    "Aarav", "Priya", "Wei", "Mei", "Mohammed", "Fatima", "Dimitri", "Elena",
    "Sione", "Aroha", "Tomas", "Bianca", "Nikolas", "Yasmin", "Rohan", "Anika",
    "Callum", "Freya", "Declan", "Imogen", "Jarrah", "Kirra", "Marcus", "Nadia",
    "Patrick", "Rosie", "Sebastian", "Talia", "Vincent", "Wren", "Xavier", "Yara",
]

LAST_NAMES = [
    "Smith", "Jones", "Williams", "Brown", "Wilson", "Taylor", "Nguyen", "Martin",
    "Anderson", "Thompson", "Walker", "Ryan", "Robinson", "Kelly", "White", "Lee",
    "Harris", "Chen", "Young", "Patel", "Singh", "Wang", "Ali", "Kaur",
    "Mitchell", "Campbell", "Stewart", "Murphy", "O'Brien", "Bennett", "Hughes", "Ellis",
    "Fletcher", "Gallagher", "Hayes", "Ingram", "Jenkins", "Kowalski", "Lambert", "Moss",
    "Novak", "Osborne", "Pritchard", "Quinn", "Rasmussen", "Sutherland", "Tran", "Underwood",
    "Vasquez", "Whitfield", "Yilmaz", "Zammit", "Abbott", "Barlow", "Chidiac", "Delaney",
]

STREETS = [
    "Anzac Pde", "Botany Rd", "Canberra Ave", "Darling St", "Elizabeth St", "Flinders Ln",
    "George St", "Hargrave St", "Ipswich Rd", "Johnston St", "King William St", "Latrobe St",
    "Macquarie St", "Northbourne Ave", "Oxford St", "Parramatta Rd", "Queen St", "Rundle Mall",
    "Sturt St", "Toorak Rd", "Unley Rd", "Victoria Pde", "Wellington St", "York St",
    "Beaufort St", "Chapel St", "Doncaster Rd", "Eagle St", "Franklin St", "Glenferrie Rd",
]

SUBURBS = [
    ("Braddon", "ACT", "2612"), ("Belconnen", "ACT", "2617"), ("Woden", "ACT", "2606"),
    ("Newtown", "NSW", "2042"), ("Parramatta", "NSW", "2150"), ("Chatswood", "NSW", "2067"),
    ("Fitzroy", "VIC", "3065"), ("Footscray", "VIC", "3011"), ("Box Hill", "VIC", "3128"),
    ("Fortitude Valley", "QLD", "4006"), ("Toowong", "QLD", "4066"), ("Southport", "QLD", "4215"),
    ("Fremantle", "WA", "6160"), ("Joondalup", "WA", "6027"), ("Norwood", "SA", "5067"),
    ("Prospect", "SA", "5082"), ("Sandy Bay", "TAS", "7005"), ("Alice Springs", "NT", "0870"),
    ("Newcastle", "NSW", "2300"), ("Geelong", "VIC", "3220"), ("Cairns", "QLD", "4870"),
]

TEAMS = ["Identity Services", "Registrations", "Renewals", "Permits", "Compliance", "Customer Care"]
ROLES = ["Officer", "Senior Officer", "Team Leader", "Supervisor"]
ACCESS_LEVELS = ["standard", "senior", "admin"]
SITES = ["Canberra", "Sydney", "Melbourne", "Brisbane", "Perth"]

QUEUES = ["Identity", "Registrations", "Renewals", "Permits", "General"]
DISPOSITIONS = ["resolved", "escalated", "callback_scheduled", "referred_to_specialist", "abandoned"]
IVR_PATHS = ["main>identity", "main>registrations", "main>renewals", "main>permits", "main>general"]

PROFILE_FIELDS = ["residential_address", "postal_address", "mobile_phone", "email",
                  "identity_document", "bank_account", "preferred_name"]

FIELD_GROUPS = {  # which CRM screens an access touched
    "profile": ["given_name", "family_name", "date_of_birth"],
    "contact": ["residential_address", "mobile_phone", "email"],
    "identity": ["identity_document", "document_number", "issue_date"],
    "financial": ["bank_account", "payment_history"],
    "linked": ["linked_customer", "household_members"],
}

USER_AGENTS = ["Chrome/Windows", "Safari/iOS", "Chrome/Android", "Edge/Windows", "Safari/macOS"]

In [4]:
def full_name():
    return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"


def street_address():
    suburb, state, postcode = random.choice(SUBURBS)
    return f"{random.randint(1, 320)} {random.choice(STREETS)}", suburb, state, postcode


def mobile_number():
    return f"04{random.randint(10, 99)} {random.randint(100, 999)} {random.randint(100, 999)}"


def date_of_birth():
    days = random.randint(18 * 365, 88 * 365)
    return (START_DATE - timedelta(days=days)).date().isoformat()


def ip_address(residential=True):
    if residential:
        return f"{random.choice([49, 58, 60, 101, 120, 139])}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
    return f"{random.choice([45, 91, 185, 194, 212])}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"


def seq_id(prefix, n):
    return f"{prefix}-{n:08d}"


def in_shift(hour, shift):
    """Is this hour inside the agent's rostered window?"""
    start, end = SHIFT_WINDOWS[shift]
    if start < end:
        return start <= hour < end
    return hour >= start or hour < end  # night shift wraps midnight


def shift_hour(shift):
    """Draw an hour from inside a rostered shift window."""
    start, end = SHIFT_WINDOWS[shift]
    hours = list(range(start, end)) if start < end else list(range(start, 24)) + list(range(0, end))
    return random.choice(hours)


def stamp(day, hour=None, shift="day"):
    hour = shift_hour(shift) if hour is None else hour
    return day.replace(hour=hour, minute=random.randint(0, 59), second=random.randint(0, 59))


def year_weight(day_index):
    """Offending grows year on year; returns a multiplier for that day's rate."""
    return (1 + FRAUD_TREND_YOY) ** (day_index // 365)

### Section 2 · Actors

Agents carry two hidden fields, `_dishonest` and `_termination_dt`, prefixed with underscores and stripped before export. Three design choices here:

- **Seniority is a mild risk factor** — senior/admin agents are 3× as likely to be selected as offenders, since broader access enables more.
- **At least two offenders are also leavers**, guaranteeing the post-termination pattern (Rule 7) exists to be found.
- **Two collusion pairs** are drawn from the offender pool. They will pull the same customers on the same days, which is what makes Rule 8's shared-customer count light up.

`protected_person` on customers (~1.2%) is a legitimate business attribute — a court order or safety concern — not a fraud flag. It raises the *severity* of an unlawful locate but doesn't identify one.

In [5]:
agents = []
for i in range(NUM_AGENTS):
    hire = START_DATE - timedelta(days=random.randint(30, 365 * 9))
    shift = random.choices(["day", "evening", "night"], weights=[0.62, 0.28, 0.10])[0]
    leaves = random.random() < P_AGENT_TERMINATED
    termination = None
    if leaves:
        # terminate part-way through the window so post-departure activity is visible
        termination = START_DATE + timedelta(days=random.randint(120, TOTAL_DAYS - 60))

    agents.append({
        "agent_id": seq_id("AGT", i + 1),
        "full_name": full_name(),
        "team": random.choice(TEAMS),
        "role": random.choices(ROLES, weights=[0.62, 0.23, 0.10, 0.05])[0],
        "access_level": random.choices(ACCESS_LEVELS, weights=[0.72, 0.21, 0.07])[0],
        "shift_pattern": shift,
        "site": random.choice(SITES),
        "hire_date": hire.date().isoformat(),
        "termination_date": termination.date().isoformat() if termination else None,
        # --- hidden, dropped before export ---
        "_dishonest": False,
        "_termination_dt": termination,
    })

# Assign the hidden propensity. Seniority is a mild risk factor (broader access).
n_bad = max(3, int(NUM_AGENTS * P_AGENT_DISHONEST))
weights = [3 if a["access_level"] in ("senior", "admin") else 1 for a in agents]
dishonest = set()
while len(dishonest) < n_bad:
    dishonest.add(random.choices(range(NUM_AGENTS), weights=weights)[0])
for i in dishonest:
    agents[i]["_dishonest"] = True

# Make sure at least two offenders also leave part-way through the window, so
# the post-termination pattern exists to be found.
leavers = [i for i in dishonest if agents[i]["_termination_dt"] is not None]
for i in list(dishonest)[:max(0, 2 - len(leavers))]:
    t = START_DATE + timedelta(days=random.randint(150, TOTAL_DAYS - 90))
    agents[i]["_termination_dt"] = t
    agents[i]["termination_date"] = t.date().isoformat()

# Two pairs of dishonest agents work together — this is what makes the collusion
# rule findable. They will share an unusual number of no-contact lookups.
bad_ids = [agents[i]["agent_id"] for i in sorted(dishonest)]
collusion_pairs = []
if len(bad_ids) >= 4:
    shuffled = bad_ids[:]
    random.shuffle(shuffled)
    collusion_pairs = [(shuffled[0], shuffled[1]), (shuffled[2], shuffled[3])]

agent_by_id = {a["agent_id"]: a for a in agents}

In [6]:
customers = []
for i in range(NUM_CUSTOMERS):
    line, suburb, state, postcode = street_address()
    customers.append({
        "customer_id": seq_id("CUS", i + 1),
        "full_name": full_name(),
        "date_of_birth": date_of_birth(),
        "residential_address": line,
        "suburb": suburb,
        "state": state,
        "postcode": postcode,
        "mobile_phone": mobile_number(),
        "email": f"{random.choice(FIRST_NAMES).lower()}.{random.choice(LAST_NAMES).lower().replace(chr(39),'')}{random.randint(1,999)}@example.com",
        "customer_since": (START_DATE - timedelta(days=random.randint(200, 365 * 12))).date().isoformat(),
        "protected_person": random.random() < 0.012,  # court order / safety concern — a real business attribute
        "linked_customer_id": None,
    })

# ~5% of customers share a household or joint registration
for c in random.sample(customers, int(NUM_CUSTOMERS * 0.05)):
    other = random.choice(customers)
    if other["customer_id"] != c["customer_id"]:
        c["linked_customer_id"] = other["customer_id"]

customer_ids = [c["customer_id"] for c in customers]
protected_ids = [c["customer_id"] for c in customers if c["protected_person"]]

### Section 3 · Row builders

Every record in the dataset comes from one of these five functions: `add_contact`, `add_access`, `add_transaction`, `add_profile_change`, `add_auth_event`.

This is the enforcement mechanism for principle 1. Legitimate and fraudulent activity call the identical builder — the only difference is the arguments. There is no `fraud=True` parameter to pass, so a label *cannot* leak into a row.

`purchase_history` is the one piece of running state: it tracks purchases/renewals per customer so that honest refunds can be attached to customers who actually bought something (94% of the time), while benefit fraud deliberately targets customers who never did.

In [7]:
contacts, crm_access, transactions, profile_changes, auth_events = [], [], [], [], []
counters = defaultdict(int)
purchase_history = defaultdict(list)  # customer_id -> [timestamps of purchases/renewals]


def next_id(prefix):
    counters[prefix] += 1
    return seq_id(prefix, counters[prefix])


def add_contact(ts, channel, customer_id, agent_id=None, queue=None, duration=None):
    cid = next_id("CON")
    contacts.append({
        "contact_id": cid,
        "contact_timestamp": ts.isoformat(sep=" "),
        "channel": channel,
        "customer_id": customer_id,
        "agent_id": agent_id,
        "queue": queue or (random.choice(QUEUES) if channel == "agent_assisted" else "self_service"),
        "ivr_path": random.choice(IVR_PATHS) if channel == "agent_assisted" else "portal>self_service",
        "duration_seconds": duration or (random.randint(90, 1100) if channel == "agent_assisted" else random.randint(40, 420)),
        "disposition": random.choices(DISPOSITIONS, weights=[0.60, 0.13, 0.12, 0.10, 0.05])[0],
        "authentication_method": random.choices(
            ["knowledge_based", "document_verification", "digital_id", "none"],
            weights=[0.55, 0.24, 0.18, 0.03])[0],
    })
    return cid


def add_access(ts, agent_id, customer_id, contact_id, screens):
    aid = next_id("ACC")
    crm_access.append({
        "access_id": aid,
        "access_timestamp": ts.isoformat(sep=" "),
        "agent_id": agent_id,
        "customer_id": customer_id,
        "contact_id": contact_id,          # NULL when there was no call in progress
        "screens_viewed": "|".join(screens),
        "records_returned": random.randint(1, 3),
        "access_channel": "crm_desktop",
    })
    return aid


def add_transaction(ts, customer_id, txn_type, amount, agent_id, contact_id, channel):
    tid = next_id("TXN")
    transactions.append({
        "transaction_id": tid,
        "transaction_timestamp": ts.isoformat(sep=" "),
        "customer_id": customer_id,
        "transaction_type": txn_type,
        "amount_aud": round(amount, 2),
        "agent_id": agent_id,
        "contact_id": contact_id,
        "channel": channel,
        "payment_method": random.choice(["card", "bpay", "direct_debit", "voucher"]),
    })
    if txn_type in ("purchase", "renewal"):
        purchase_history[customer_id].append(ts)
    return tid


def add_profile_change(ts, customer_id, fields, agent_id, contact_id, channel):
    for field in fields:
        profile_changes.append({
            "change_id": next_id("CHG"),
            "change_timestamp": ts.isoformat(sep=" "),
            "customer_id": customer_id,
            "field_changed": field,
            "previous_value": "REDACTED",
            "new_value": "REDACTED",
            "agent_id": agent_id,
            "contact_id": contact_id,
            "channel": channel,
        })


def add_auth_event(ts, customer_id, event_type, ip=None, ua=None):
    auth_events.append({
        "auth_event_id": next_id("AUT"),
        "event_timestamp": ts.isoformat(sep=" "),
        "customer_id": customer_id,
        "event_type": event_type,
        "source_ip": ip or ip_address(),
        "user_agent": ua or random.choice(USER_AGENTS),
        "channel": "self_service_portal",
    })


def active_agents_on(day):
    """Agents employed on this date. Used for legitimate rostering only."""
    return [a for a in agents if a["_termination_dt"] is None or a["_termination_dt"] > day]

### Section 4 · Activity engine

One chronological pass over three years. Each day:

**Ordinary business runs first** — calls, portal sessions, lookups, transactions, profile changes, honest failed logins. This block also plants the noise that keeps the rules honest: legitimate warm transfers (agent opens a second customer mid-call), QA/callback-prep lookups with no call attached, honest bulk profile updates when someone moves house, and password fumbles.

**Then propensity-driven behaviour**, through the same builders:

| | Pattern | Data footprint |
|---|---|---|
| A | Unlawful locate | Lookup with no `contact_id`, weighted toward protected persons, sometimes outside the roster |
| B | Excessive lookups | 22–40 unrelated records inside one hour |
| C | Benefit fraud | Refund/waiver for a customer with no purchase history, at 5–10× honest refund values |
| D | Identity takeover | 4–5 fields rewritten in one contact, always including the ID document |
| E | Collusion | Paired agents pulling the same customers, neither on a call |
| F | Account takeover | Failed-login run from a hosting-range IP → success → phone/email change → transaction |
| G | Post-termination | Lookups weeks after the officer's last day |

Note what's *absent*: no row is marked, no side table records which rows were planted. The behaviour is the only evidence.

In [8]:
for day_index, day in enumerate(DATE_RANGE):
    weekday = day.weekday() < 5
    volume = int(CONTACTS_PER_DAY * (1.0 if weekday else 0.35))
    roster = active_agents_on(day)
    if not roster:
        continue
    trend = year_weight(day_index)

    # ---------------------------------------------------------------- ordinary
    for _ in range(volume):
        customer_id = random.choice(customer_ids)
        agent_assisted = random.random() > P_SELF_SERVICE

        if agent_assisted:
            agent = random.choice(roster)
            ts = stamp(day, shift=agent["shift_pattern"])
            contact_id = add_contact(ts, "agent_assisted", customer_id, agent["agent_id"])

            # the lookups that go with the call — pretext satisfied. An agent
            # typically opens two or three screens while handling one contact.
            for k in range(random.randint(1, 3)):
                add_access(ts + timedelta(seconds=random.randint(0, 600)),
                           agent["agent_id"], customer_id, contact_id,
                           random.sample(list(FIELD_GROUPS), k=random.randint(1, 3)))

            # legitimate warm transfer: agent opens a second customer's record
            if random.random() < P_LEGIT_WARM_TRANSFER:
                add_access(ts + timedelta(minutes=random.randint(1, 6)), agent["agent_id"],
                           random.choice(customer_ids), contact_id, ["profile", "contact"])

            if random.random() < P_CONTACT_HAS_TRANSACTION:
                if random.random() < P_TXN_IS_REFUND:
                    # honest refunds overwhelmingly follow a real purchase
                    if purchase_history[customer_id] and random.random() < P_LEGIT_REFUND_HAS_PURCHASE:
                        add_transaction(ts, customer_id, random.choice(["refund", "fee_waiver"]),
                                        random.uniform(25, 320), agent["agent_id"], contact_id, "agent_assisted")
                else:
                    add_transaction(ts, customer_id, random.choice(["purchase", "renewal"]),
                                    random.uniform(20, 450), agent["agent_id"], contact_id, "agent_assisted")

            if random.random() < P_CONTACT_HAS_PROFILE_CHANGE:
                # honest updates touch one field, or two when someone moves house
                if random.random() < P_LEGIT_3_FIELD_CHANGE:
                    # an honest bulk update is a change of circumstances: new
                    # address, new phone, new email. It never rewrites the
                    # identity document or bank account in the same breath.
                    fields = random.sample(
                        ["residential_address", "postal_address", "mobile_phone",
                         "email", "preferred_name"], 3)
                elif random.random() < 0.30:
                    fields = ["residential_address", "postal_address"]
                else:
                    fields = [random.choice(PROFILE_FIELDS)]
                add_profile_change(ts, customer_id, fields, agent["agent_id"], contact_id, "agent_assisted")

        else:
            ts = stamp(day, hour=random.randint(6, 23))
            ip = ip_address()
            ua = random.choice(USER_AGENTS)

            # honest people mistype their password all the time
            if random.random() < P_SELF_SERVE_LOCKOUT:
                n_fail = random.randint(3, 5)
            elif random.random() < P_SELF_SERVE_FAILED_LOGIN:
                n_fail = random.randint(1, 2)
            else:
                n_fail = 0
            for k in range(n_fail):
                add_auth_event(ts - timedelta(minutes=n_fail - k), customer_id, "failed_login", ip, ua)
            add_auth_event(ts, customer_id, "login_success", ip, ua)

            contact_id = add_contact(ts, "self_service", customer_id)

            if random.random() < P_CONTACT_HAS_TRANSACTION:
                if random.random() < P_TXN_IS_REFUND and purchase_history[customer_id]:
                    add_transaction(ts, customer_id, "refund", random.uniform(25, 200), None, contact_id, "self_service")
                else:
                    add_transaction(ts, customer_id, random.choice(["purchase", "renewal"]),
                                    random.uniform(20, 450), None, contact_id, "self_service")

            if random.random() < P_CONTACT_HAS_PROFILE_CHANGE * 0.5:
                add_profile_change(ts, customer_id, [random.choice(PROFILE_FIELDS)], None, contact_id, "self_service")

    # legitimate lookups with no call attached — QA sampling, callback prep,
    # complaint handling. These are why "no contact_id" is not itself damning.
    for _ in range(int(volume * P_LEGIT_NO_CONTACT_ACCESS)):
        agent = random.choice(roster)
        add_access(stamp(day, shift=agent["shift_pattern"]), agent["agent_id"],
                   random.choice(customer_ids), None, ["profile"])

    # ------------------------------------------------------- driven by propensity
    for agent in roster:
        if not agent["_dishonest"]:
            continue
        aid = agent["agent_id"]

        # (A) Unlawful locate — looking up people with no business reason.
        #     Some during rostered hours, some late at night.
        if random.random() < (LOCATE_LOOKUPS_PER_MONTH * trend) / 30:
            target = random.choice(protected_ids) if (protected_ids and random.random() < 0.35) \
                else random.choice(customer_ids)
            if random.random() < P_LOCATE_AFTER_HOURS:
                hours = [h for h in range(24) if not in_shift(h, agent["shift_pattern"])]
                ts = stamp(day, hour=random.choice(hours))
            else:
                ts = stamp(day, shift=agent["shift_pattern"])
            add_access(ts, aid, target, None, ["contact", "linked", "profile"])

        # (B) Excessive lookups — a burst of unrelated records in one hour.
        if random.random() < (VELOCITY_BURSTS_PER_YEAR * trend) / 365:
            base = stamp(day, shift=agent["shift_pattern"])
            for _ in range(random.randint(*VELOCITY_BURST_SIZE)):
                add_access(base + timedelta(minutes=random.randint(0, 55)), aid,
                           random.choice(customer_ids), None, ["contact", "linked"])

        # (C) Benefit fraud — a refund or waiver processed for a customer who
        #     never paid anything. Built by the same transaction function.
        if random.random() < (BENEFIT_FRAUD_PER_MONTH * trend) / 30:
            target = random.choice([c for c in customer_ids if not purchase_history[c]] or customer_ids)
            ts = stamp(day, shift=agent["shift_pattern"])
            contact_id = add_contact(ts, "agent_assisted", target, aid)
            add_access(ts, aid, target, contact_id, ["financial", "profile"])
            add_transaction(ts, target, random.choice(["refund", "fee_waiver"]),
                            random.uniform(180, 2400), aid, contact_id, "agent_assisted")

        # (D) Identity takeover at the counter — an agent rewrites most of a
        #     customer record in a single contact, including the ID document.
        if random.random() < (IDENTITY_FRAUD_PER_MONTH * trend) / 30:
            target = random.choice(customer_ids)
            ts = stamp(day, shift=agent["shift_pattern"])
            contact_id = add_contact(ts, "agent_assisted", target, aid)
            add_access(ts, aid, target, contact_id, ["identity", "contact", "profile"])
            fields = random.sample(["residential_address", "mobile_phone", "email",
                                    "identity_document", "bank_account"], random.randint(4, 5))
            add_profile_change(ts, target, fields, aid, contact_id, "agent_assisted")

    # (E) Collusion — paired agents pull the same customers on the same day,
    #     neither of them on a call. Emerges as an abnormal shared-customer count.
    for a1, a2 in collusion_pairs:
        if a1 not in {a["agent_id"] for a in roster} or a2 not in {a["agent_id"] for a in roster}:
            continue
        if random.random() < 0.04 * trend:
            shared = random.sample(customer_ids, random.randint(2, 4))
            base = stamp(day, shift=agent_by_id[a1]["shift_pattern"])
            for cust in shared:
                add_access(base, a1, cust, None, ["contact", "linked"])
                add_access(base + timedelta(minutes=random.randint(5, 180)), a2, cust, None, ["contact", "linked"])

    # (F) External account takeover — a run of failed logins from an unfamiliar
    #     IP, then a success, then a valuable change. Same builders as above.
    if random.random() < (ACCOUNT_TAKEOVERS_PER_MONTH * trend) / 30:
        victim = random.choice(customer_ids)
        ts = stamp(day, hour=random.choice([0, 1, 2, 3, 4, 22, 23]))
        attacker_ip = ip_address(residential=False)
        for k in range(random.randint(4, 9)):
            add_auth_event(ts - timedelta(minutes=12 - k), victim, "failed_login", attacker_ip, "Chrome/Linux")
        add_auth_event(ts, victim, "login_success", attacker_ip, "Chrome/Linux")
        contact_id = add_contact(ts, "self_service", victim)
        add_profile_change(ts + timedelta(minutes=random.randint(1, 20)), victim,
                           ["mobile_phone", "email"], None, contact_id, "self_service")
        add_transaction(ts + timedelta(minutes=random.randint(2, 40)), victim,
                        random.choice(["renewal", "replacement_document"]),
                        random.uniform(40, 300), None, contact_id, "self_service")

# (G) Post-termination access — a departed officer's credentials are still live.
for agent in agents:
    if agent["_termination_dt"] is None or not agent["_dishonest"]:
        continue
    for _ in range(random.randint(*POST_TERMINATION_ACCESSES)):
        day = agent["_termination_dt"] + timedelta(days=random.randint(2, 70))
        if day > DATE_RANGE[-1]:
            continue
        add_access(stamp(day, hour=random.randint(19, 23)), agent["agent_id"],
                   random.choice(customer_ids), None, ["contact", "linked"])

### Section 5 · Export

The hidden columns are dropped, and a guard rail asserts that no exported column name contains `fraud`, `label`, `scenario`, `suspicious`, `anomal`, or `risk_score` — the build fails loudly if a label ever creeps back in.

Expect roughly: 134k contacts, 150k CRM accesses, 30k transactions, 12k profile changes, 79k auth events (~45 MB).

In [9]:
df_agents = pd.DataFrame(agents).drop(columns=["_dishonest", "_termination_dt"])
df_customers = pd.DataFrame(customers)
df_contacts = pd.DataFrame(contacts).sort_values("contact_timestamp")
df_access = pd.DataFrame(crm_access).sort_values("access_timestamp")
df_transactions = pd.DataFrame(transactions).sort_values("transaction_timestamp")
df_changes = pd.DataFrame(profile_changes).sort_values("change_timestamp")
df_auth = pd.DataFrame(auth_events).sort_values("event_timestamp")

TABLES = {
    "agents": df_agents,
    "customers": df_customers,
    "contacts": df_contacts,
    "crm_access": df_access,
    "transactions": df_transactions,
    "profile_changes": df_changes,
    "auth_events": df_auth,
}

# Guard rail: fail loudly if a label ever creeps back into the output.
BANNED = ("fraud", "label", "scenario", "suspicious", "anomal", "is_bad", "risk_score")
for name, df in TABLES.items():
    for col in df.columns:
        assert not any(b in col.lower() for b in BANNED), f"label leak: {name}.{col}"
    assert not col.startswith("_"), f"internal column exported: {name}.{col}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
for name, df in TABLES.items():
    df.to_csv(os.path.join(OUTPUT_DIR, f"{name}.csv"), index=False)

print(f"Written to {OUTPUT_DIR}/  — no fraud labels present\n")
for name, df in TABLES.items():
    print(f"  {name:<18} {len(df):>8,} rows  x {len(df.columns):>2} cols")

print(f"\n  Period            {DATE_RANGE[0].date()} to {DATE_RANGE[-1].date()}")
print(f"  Refunds/waivers   {(df_transactions['transaction_type'].isin(['refund','fee_waiver'])).sum():,} "
      f"({(df_transactions['transaction_type'].isin(['refund','fee_waiver'])).mean():.1%} of transactions)")
print(f"  Access w/o contact {df_access['contact_id'].isna().sum():,} "
      f"({df_access['contact_id'].isna().mean():.1%} of lookups)")
print(f"  Failed logins     {(df_auth['event_type']=='failed_login').sum():,}")

Written to output/  — no fraud labels present

  agents                   20 rows  x  9 cols
  customers             7,500 rows  x 12 cols
  contacts            133,992 rows  x 10 cols
  crm_access          149,645 rows  x  8 cols
  transactions         29,557 rows  x  9 cols
  profile_changes      12,186 rows  x  9 cols
  auth_events          79,184 rows  x  7 cols

  Period            2023-01-01 to 2025-12-30
  Refunds/waivers   1,615 (5.5% of transactions)
  Access w/o contact 1,893 (1.3% of lookups)
  Failed logins     18,874


## Part 2 — Detection

The generator's variables above still hold the hidden truth, so from here on we **reload the CSVs from disk** and use nothing else. This is the analyst's view: seven tables, no labels.

Seven of the rules are a single query; Rule 8 needs a volume adjustment and is imported from the package. Watch two numbers per rule — the hit count, and the count of *near-misses correctly left alone*. The second number is what proves the base rates are doing their job.

In [10]:
# Reload the exported CSVs fresh, exactly as an analyst would receive them.
# From here on we use ONLY what is on disk — none of the in-memory generator
# state (and none of its hidden variables) is available to the rules.

D = OUTPUT_DIR
agents_t = pd.read_csv(f"{D}/agents.csv")
customers_t = pd.read_csv(f"{D}/customers.csv")
contacts_t = pd.read_csv(f"{D}/contacts.csv", parse_dates=["contact_timestamp"])
access_t = pd.read_csv(f"{D}/crm_access.csv", parse_dates=["access_timestamp"])
txns_t = pd.read_csv(f"{D}/transactions.csv", parse_dates=["transaction_timestamp"])
changes_t = pd.read_csv(f"{D}/profile_changes.csv", parse_dates=["change_timestamp"])
auth_t = pd.read_csv(f"{D}/auth_events.csv", parse_dates=["event_timestamp"])

SHIFTS = {"day": (7, 15), "evening": (15, 23), "night": (23, 7)}

def hdr(n, title):
    print(f"{'='*68}\nRULE {n}: {title}\n{'='*68}")

### Rule 1 · Lookup with no matching call in progress

The pretext check: left-join `crm_access` to `contacts` and flag rows where no call was in progress, or the call was with a *different* customer.

This rule fires on every agent — warm transfers and QA reviews legitimately produce these rows. That's deliberate. The signal is in the **ranking**: offenders sit at several hundred lookups where the honest pack tops out around 80. A rule that fired only on the guilty would be a label wearing a costume.

In [11]:
# --- R1: Unlawful locate -----------------------------------------------------
hdr(1, "Lookup with no matching call in progress")
a = access_t.merge(contacts_t[["contact_id", "customer_id"]].rename(columns={"customer_id": "contact_customer"}),
                 on="contact_id", how="left")
r1 = a[a["contact_id"].isna() | (a["contact_customer"] != a["customer_id"])]
print(f"  hits: {len(r1):,}  ({len(r1)/len(access_t):.2%} of lookups)")
print(f"  agents_t implicated: {r1['agent_id'].nunique()} of {len(agents_t)}")
top = r1.groupby("agent_id").size().sort_values(ascending=False)
print(f"  top 5 agents_t: {list(top.head(5).items())}")

RULE 1: Lookup with no matching call in progress
  hits: 2,303  (1.54% of lookups)
  agents_t implicated: 20 of 20
  top 5 agents_t: [('AGT-00000011', 903), ('AGT-00000010', 239), ('AGT-00000004', 81), ('AGT-00000009', 78), ('AGT-00000012', 77)]


### Rule 2 · Lookup outside the rostered shift

Join to `agents.shift_pattern` and flag accesses **more than an hour** outside the roster window. The grace hour matters: without it, agents finishing case notes after shift end swamp the rule with false positives.

In [12]:
# --- R2: After-hours access_t --------------------------------------------------
hdr(2, "Lookup more than an hour outside the agent's rostered shift")
a2 = access_t.merge(agents_t[["agent_id", "shift_pattern"]], on="agent_id", how="left")
a2["hour"] = a2["access_timestamp"].dt.hour


def outside(row):
    """One hour of grace either side of the roster — agents_t finish case notes."""
    s, e = SHIFTS[row["shift_pattern"]]
    s, e = (s - 1) % 24, (e + 1) % 24
    return not (s <= row["hour"] < e) if s < e else not (row["hour"] >= s or row["hour"] < e)


r2 = a2[a2.apply(outside, axis=1)]
print(f"  hits: {len(r2):,}  ({len(r2)/len(access_t):.2%} of lookups)")
print(f"  agents_t implicated: {r2['agent_id'].nunique()}")

RULE 2: Lookup more than an hour outside the agent's rostered shift


  hits: 221  (0.15% of lookups)
  agents_t implicated: 3


### Rule 3 · Excessive lookups in one hour

Distinct customers viewed per agent-hour. The threshold of 10 is derived from the data, not guessed: the honest 99.9th percentile is ~5 customers per hour, and the planted bursts reach the mid-30s. In a client conversation, deriving the threshold from the baseline *is* the methodology point.

In [13]:
# --- R3: Excessive lookups ---------------------------------------------------
hdr(3, "More than 10 distinct customers_t viewed by one agent in one hour")
a3 = access_t.copy()
a3["hour_bucket"] = a3["access_timestamp"].dt.floor("h")
v = a3.groupby(["agent_id", "hour_bucket"])["customer_id"].nunique().reset_index(name="customers_viewed")
r3 = v[v["customers_viewed"] > 10]
print(f"  hits: {len(r3):,} agent-hours")
print(f"  agents_t implicated: {r3['agent_id'].nunique()}")
print(f"  max in an hour: {v['customers_viewed'].max()}  |  95th pct overall: {v['customers_viewed'].quantile(0.95):.0f}")

RULE 3: More than 10 distinct customers_t viewed by one agent in one hour
  hits: 20 agent-hours
  agents_t implicated: 2
  max in an hour: 34  |  95th pct overall: 2


### Rule 4 · Refund with no prior purchase

An anti-join: refunds and fee waivers for customers with no earlier purchase or renewal. Two things to show a client here — the hit rate (~6% of refunds, so `transaction_type` alone carries no signal), and the value gap: median flagged amount ~$1,370 against ~$140 for clean refunds.

In [14]:
# --- R4: Refund with no prior purchase ---------------------------------------
hdr(4, "Refund or fee waiver with no earlier purchase by that customer")
refunds = txns_t[txns_t["transaction_type"].isin(["refund", "fee_waiver"])]
purchases = txns_t[txns_t["transaction_type"].isin(["purchase", "renewal"])]
first_purchase = purchases.groupby("customer_id")["transaction_timestamp"].min()
r4 = refunds[~refunds["customer_id"].map(first_purchase).notna() |
             (refunds["transaction_timestamp"] < refunds["customer_id"].map(first_purchase))]
print(f"  hits: {len(r4):,}  ({len(r4)/len(refunds):.1%} of all refunds)")
print(f"  median amount flagged: ${r4['amount_aud'].median():,.0f}  vs clean refunds ${refunds[~refunds.index.isin(r4.index)]['amount_aud'].median():,.0f}")
print(f"  agents_t implicated: {r4['agent_id'].nunique()}")

RULE 4: Refund or fee waiver with no earlier purchase by that customer
  hits: 99  (6.1% of all refunds)
  median amount flagged: $1,371  vs clean refunds $137
  agents_t implicated: 3


### Rule 5 · Bulk identity rewrite

Three or more fields changed in a single contact **including the identity document**. The second condition is what buys precision: honest customers bulk-update address, phone and email when they move house — about a hundred such contacts exist and are correctly left alone — but they never rewrite the ID document in the same breath.

In [15]:
# --- R5: Bulk identity rewrite -----------------------------------------------
hdr(5, "Three or more profile fields changed in one contact, including the ID document")
g = changes_t.groupby("contact_id")["field_changed"].nunique()
has_id = set(changes_t[changes_t["field_changed"] == "identity_document"]["contact_id"])
r5_contacts = [c for c in g[g >= 3].index if c in has_id]
r5 = changes_t[changes_t["contact_id"].isin(r5_contacts)]
print(f"  hits: {len(r5_contacts):,} contacts_t ({len(r5_contacts)/changes_t['contact_id'].nunique():.1%} of contacts_t with changes_t)")
print(f"  contacts_t with 3+ fields but no ID change (not flagged): {(g >= 3).sum() - len(r5_contacts):,}")
print(f"  agents_t implicated: {r5['agent_id'].nunique()}")

RULE 5: Three or more profile fields changed in one contact, including the ID document
  hits: 48 contacts_t (0.5% of contacts_t with changes_t)
  contacts_t with 3+ fields but no ID change (not flagged): 102
  agents_t implicated: 3


### Rule 6 · Account takeover

A conjunction of three events: ≥3 failed logins within 15 minutes → successful login → a profile change within the hour. Over a thousand sessions have the failed-login run *without* the follow-up change (people genuinely forget passwords) and are not flagged. The gap between those two numbers is the rule earning its keep.

In [16]:
# --- R6: Account takeover ----------------------------------------------------
hdr(6, "Three or more failed logins, then success, then a change within the hour")
auth_s = auth_t.sort_values("event_timestamp")
fails = auth_s[auth_s["event_type"] == "failed_login"]
success = auth_s[auth_s["event_type"] == "login_success"]
hits = []
fails_by_cust = {k: v["event_timestamp"].values for k, v in fails.groupby("customer_id")}
chg_by_cust = {k: v["change_timestamp"].values for k, v in changes_t.groupby("customer_id")}
for _, s in success.iterrows():
    f = fails_by_cust.get(s["customer_id"])
    if f is None:
        continue
    window = ((f < s["event_timestamp"].to_datetime64()) &
              (f > (s["event_timestamp"] - pd.Timedelta(minutes=15)).to_datetime64())).sum()
    if window < 3:
        continue
    c = chg_by_cust.get(s["customer_id"])
    if c is None:
        continue
    after = ((c > s["event_timestamp"].to_datetime64()) &
             (c < (s["event_timestamp"] + pd.Timedelta(minutes=60)).to_datetime64())).sum()
    if after:
        hits.append(s["customer_id"])
print(f"  hits: {len(hits):,} sessions")
print(f"  sessions with 3+ failures but no follow-up change (not flagged): "
      f"{sum(1 for _, s in success.iterrows() if (lambda f: f is not None and ((f < s['event_timestamp'].to_datetime64()) & (f > (s['event_timestamp']-pd.Timedelta(minutes=15)).to_datetime64())).sum() >= 3)(fails_by_cust.get(s['customer_id']))) - len(hits):,}")

RULE 6: Three or more failed logins, then success, then a change within the hour


  hits: 266 sessions


  sessions with 3+ failures but no follow-up change (not flagged): 1,176


### Rule 7 · Post-termination access

The simplest rule in the set and the strongest demo moment: lookups timestamped after the agent's termination date. Two former officers, credentials still live.

In [17]:
# --- R7: Post-termination access_t ---------------------------------------------
hdr(7, "Lookup after the agent's termination date")
a7 = access_t.merge(agents_t[["agent_id", "termination_date"]], on="agent_id", how="left")
a7["termination_date"] = pd.to_datetime(a7["termination_date"])
r7 = a7[a7["termination_date"].notna() & (a7["access_timestamp"] > a7["termination_date"])]
print(f"  hits: {len(r7):,}")
print(f"  agents_t implicated: {r7['agent_id'].nunique()} of {agents_t['termination_date'].notna().sum()} leavers")

RULE 7: Lookup after the agent's termination date
  hits: 14
  agents_t implicated: 2 of 4 leavers


### Rule 8 · Collusion pairs

Self-join no-call lookups on `customer_id` and count distinct shared customers per agent pair — then **normalise by volume**, which is the whole point of this rule.

The raw shared count is dominated by how much each agent looks up at all. Two offenders each making several hundred unlawful locates across the same 7,500 customers overlap heavily by chance: at this configuration the highest raw count belongs to a pair that never colluded, sharing 137 customers against an expectation of 78. A planted pair sits second on 106 against an expectation of 38.

So compare each pair's overlap to what chance predicts for two agents of those volumes — a hypergeometric expectation — and divide by its standard deviation. Ranked that way the two planted pairs come first and second out of 190, with a clear gap to third.

That normalisation is the methodology point worth making to a client. The obvious version of this rule mostly restates Rule 1: it finds agents who do a lot of no-call lookups, which Rule 1 already told you. The scoring lives in `fraud_generator.detection` rather than in this cell, because a rule this easy to get wrong needs tests.

In [ ]:
# --- R8: Collusion -----------------------------------------------------------
hdr(8, "Agent pairs sharing an abnormal overlap in no-call lookups")

from fraud_generator.detection import collusion_pair_scores

scores = collusion_pair_scores(
    access_t.to_dict("records"),
    customers_t["customer_id"].nunique(),
)
r8 = pd.DataFrame(
    [(s.pair[0], s.pair[1], s.shared, s.expected, s.z) for s in scores],
    columns=["agent_x", "agent_y", "shared", "expected", "z"],
)

top_raw = r8.loc[r8["shared"].idxmax()]
print(f"  pairs with 3+ shared customers: {len(r8)}")
print(f"  highest RAW count:  {top_raw.agent_x} / {top_raw.agent_y}  "
      f"{int(top_raw.shared)} shared (expected {top_raw.expected:.0f}) -> z={top_raw.z:.1f}")
print()
print("  ranked by excess over chance:")
print(r8.head(6).to_string(index=False, float_format="%.1f"))

### Convergence — the punchline

Stack the rules per agent. Expect three agents tripping five or six independent rules while the other seventeen trip one — and only Rule 1, the noisy one. No single rule is decisive; the *convergence* is. That's a better story than a rule that works perfectly, and it's how real insider-threat programs actually operate.

In [19]:
# --- Overlap: how many agents_t does the full rule set converge on? -------------
print(f"\n{'='*68}\nCONVERGENCE\n{'='*68}")
flagged = pd.concat([
    r1.groupby("agent_id").size().rename("R1_no_pretext"),
    r2.groupby("agent_id").size().rename("R2_after_hours"),
    r3.groupby("agent_id").size().rename("R3_velocity"),
    r4.groupby("agent_id").size().rename("R4_refund"),
    r5.groupby("agent_id").size().rename("R5_identity"),
    r7.groupby("agent_id").size().rename("R7_terminated"),
], axis=1).fillna(0)
flagged["rules_tripped"] = (flagged > 0).sum(axis=1)
print(flagged.sort_values("rules_tripped", ascending=False).head(12).to_string())
print(f"\n  agents tripping 3+ rules: {(flagged['rules_tripped'] >= 3).sum()} of {len(agents_t)}")


CONVERGENCE
              R1_no_pretext  R2_after_hours  R3_velocity  R4_refund  R5_identity  R7_terminated  rules_tripped
agent_id                                                                                                      
AGT-00000010            239            32.0          7.0       10.0         45.0            9.0              6
AGT-00000001             67            27.0          0.0       12.0         33.0            5.0              5
AGT-00000011            903           162.0         13.0       77.0        146.0            0.0              5
AGT-00000002             11             0.0          0.0        0.0          0.0            0.0              1
AGT-00000004             81             0.0          0.0        0.0          0.0            0.0              1
AGT-00000003             63             0.0          0.0        0.0          0.0            0.0              1
AGT-00000005             74             0.0          0.0        0.0          0.0            0.0    

---

## Tuning cheat-sheet

| Goal | Change |
|---|---|
| Bigger dataset | `NUM_CUSTOMERS`, `CONTACTS_PER_DAY`, `YEARS` |
| Harder to detect | Raise `P_LEGIT_NO_CONTACT_ACCESS`, `P_LEGIT_WARM_TRANSFER`, `P_LEGIT_3_FIELD_CHANGE`, `P_SELF_SERVE_LOCKOUT` |
| Easier to detect | Raise `LOCATE_LOOKUPS_PER_MONTH`, `BENEFIT_FRAUD_PER_MONTH`, `VELOCITY_BURSTS_PER_YEAR` |
| More / fewer offenders | `P_AGENT_DISHONEST` |
| Fresh dataset, same shape | Change `SEED` |

**Caveat before client use:** contact volume is a sampled extract, not a full production feed — agents here handle fewer contacts per day than real officers. Rule thresholds are calibrated against *this* dataset's baseline; recalibrate before quoting them as operational thresholds.